# Notebook 2 — `training_normalizer.ipynb`

**Sovereign Dialect-Bridge · Step 2 — Train mT5-small Normalizer (Stage 1)**

Fine-tune **mT5-small** untuk menerjemahkan kalimat dialek (Jawa/Sunda/Minang/dll.) → Bahasa Indonesia baku. Output disimpan di `models/normalizer/`.

**Sumber data (gabungan, ordered by priority):**

| Sumber | Volume | Kualitas | Dialek |
|--------|--------|----------|--------|
| `GEM/indonlg` (mt_id_jv, mt_id_su) | 5–10K/dialek | ⭐⭐⭐ human-translated, EMNLP 2021 | Jawa, Sunda |
| `Exqrch/IndonesianNMT` (id_jav, id_sun, id_min) | 10K–100K/dialek | ⭐⭐ besar | Jawa, Sunda, Minang |
| `indonlp/NusaX-MT` | 1K total | ⭐⭐⭐ native speaker | 10 dialek lain (backup) |

**Target:** BLEU-4 > 20 pada validation NusaX.
**Hardware:** RTX 3090/4090 (24 GB), bf16, ~45 menit total.


## 1. Setup environment


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]   = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc, json, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = torch.cuda.is_bf16_supported() if DEVICE == "cuda" else False
USE_FP16 = False   # JANGAN pakai fp16 — selalu bf16 atau fp32

assert DEVICE == "cuda", "Notebook ini butuh GPU (vast.ai RTX 3090/4090)."
print(f"GPU   : {torch.cuda.get_device_name(0)}")
print(f"VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"bf16  : {USE_BF16}  |  fp16: {USE_FP16}")


## 2. Install dependencies

```bash
pip install transformers==4.40.0 datasets accelerate sentencepiece sacrebleu sacremoses
```


## 3. Configuration (sinkron CLAUDE.md)


In [ ]:
# Auto-detect project root
CWD = Path.cwd()
if (CWD / "dataset").exists():
    ROOT = CWD
elif (CWD.parent / "dataset").exists():
    ROOT = CWD.parent
else:
    ROOT = CWD

MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR   = ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

NORM_MODEL         = "google/mt5-small"
NORM_MAX_INPUT     = 256
NORM_MAX_TARGET    = 256
NORM_LR            = 5e-5
NORM_EPOCHS        = 5
NORM_BATCH         = 8
NORM_GRAD_ACCUM    = 2
NORM_MAX_GRAD_NORM = 1.0
NORM_OUTPUT_DIR    = str(MODELS_DIR / "normalizer")
RANDOM_SEED        = 42

random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
print(f"ROOT          : {ROOT}")
print(f"NORM_OUTPUT   : {NORM_OUTPUT_DIR}")
print(f"effective batch = {NORM_BATCH * NORM_GRAD_ACCUM} (batch={NORM_BATCH}, grad_accum={NORM_GRAD_ACCUM})")


## 4. Load datasets

Tiga sumber digabung. Jika sumber online gagal di-load (network issue di vast.ai), notebook tetap lanjut dengan sumber yang tersedia.


In [ ]:
from datasets import load_dataset

mt_data = []

# 1. IndoNLG MT — prioritas tertinggi (human-translated)
INDONLG_CONFIGS = [("mt_id_jv", "javanese"), ("mt_id_su", "sundanese")]
for cfg, dialect in INDONLG_CONFIGS:
    try:
        ds = load_dataset("GEM/indonlg", cfg, trust_remote_code=True)
        for split in ds.keys():
            for row in ds[split]:
                # IndoNLG: 'gem_id', 'target' (BI), 'references' (list of dialect)
                bi   = str(row.get("target") or "")
                refs = row.get("references") or []
                dial = str(refs[0]) if refs else ""
                if bi and dial:
                    mt_data.append({"indonesian": bi, "dialect": dial,
                                    "dialect_name": dialect, "source": "indonlg"})
        print(f"  ✓ IndoNLG {cfg:10s} loaded")
    except Exception as e:
        print(f"  ✗ IndoNLG {cfg:10s} failed: {e}")

print(f"After IndoNLG : {len(mt_data):,} pairs")


In [ ]:
# 2. IndonesianNMT — volume besar
NMT_SUBSETS = [("id_jav", "javanese"), ("id_sun", "sundanese"), ("id_min", "minangkabau")]
for subset, dialect in NMT_SUBSETS:
    try:
        ds = load_dataset("Exqrch/IndonesianNMT", subset, trust_remote_code=True)
        n_before = len(mt_data)
        for split in ds.keys():
            for row in ds[split]:
                cols = list(row.keys())
                bi_col   = next((c for c in cols if c.lower() in {"ind","id","indonesian","ind_id","idn"}), None)
                dial_col = next((c for c in cols if c != bi_col), None)
                if bi_col and dial_col:
                    bi = str(row[bi_col] or "")
                    dl = str(row[dial_col] or "")
                    if bi and dl:
                        mt_data.append({"indonesian": bi, "dialect": dl,
                                        "dialect_name": dialect, "source": "indonesian_nmt"})
        print(f"  ✓ IndonesianNMT {subset:8s} added {len(mt_data) - n_before:,}")
    except Exception as e:
        print(f"  ✗ IndonesianNMT {subset:8s} failed: {e}")

print(f"After IndonesianNMT : {len(mt_data):,} pairs")


In [ ]:
# 3. NusaX-MT — semua dialek lain (backup), juga dipakai untuk evaluasi & dialect dict
df_nusax = None
try:
    nusax = load_dataset("indonlp/NusaX-MT", trust_remote_code=True)
    train_split = nusax.get("train") or nusax[list(nusax.keys())[0]]
    cols = train_split.column_names
    bi_col = next((c for c in cols if c.lower() in {"indonesian","ind","id"}), None)
    DIALECT_COLS = [c for c in cols if c not in (bi_col, "english", "id", "gem_id", "index")]
    n_before = len(mt_data)
    for row in train_split:
        bi = str(row.get(bi_col) or "")
        for d in DIALECT_COLS:
            dial = str(row.get(d) or "")
            if bi and dial:
                mt_data.append({"indonesian": bi, "dialect": dial,
                                "dialect_name": d, "source": "nusax"})
    print(f"  ✓ NusaX-MT loaded, added {len(mt_data) - n_before:,} pairs across {len(DIALECT_COLS)} dialects")
    # Simpan dataframe NusaX untuk dipakai di sel selanjutnya (auto-expand dict + eval)
    df_nusax = nusax["train"].to_pandas() if "train" in nusax else train_split.to_pandas()
except Exception as e:
    print(f"  ✗ NusaX-MT failed: {e} — fallback ke CSV lokal")
    # Fallback: load dari lokal
    local = ROOT / "dataset" / "nusax" / "datasets" / "mt"
    if local.exists():
        df_nusax = pd.concat([pd.read_csv(local / f) for f in ["train.csv","valid.csv","test.csv"]
                              if (local / f).exists()], ignore_index=True)
        cols = df_nusax.columns.tolist()
        bi_col = next((c for c in cols if c.lower() in {"indonesian","ind","id"}), "indonesian")
        DIALECT_COLS = [c for c in cols if c not in (bi_col, "english", "Unnamed: 0", "id")]
        n_before = len(mt_data)
        for _, row in df_nusax.iterrows():
            bi = str(row.get(bi_col) or "")
            for d in DIALECT_COLS:
                dial = str(row.get(d) or "")
                if bi and dial:
                    mt_data.append({"indonesian": bi, "dialect": dial,
                                    "dialect_name": d, "source": "nusax_local"})
        print(f"  ✓ NusaX local fallback added {len(mt_data) - n_before:,}")

print(f"\nTOTAL MT pairs: {len(mt_data):,}")


## 5. Filter & split MT pairs


In [ ]:
df_mt = pd.DataFrame(mt_data).dropna(subset=["indonesian", "dialect"])
df_mt = df_mt[df_mt["indonesian"].str.strip().astype(bool) & df_mt["dialect"].str.strip().astype(bool)]

# Min 3 kata di kedua sisi (filter pasangan terlalu pendek)
df_mt = df_mt[
    (df_mt["dialect"].str.split().str.len()    >= 3) &
    (df_mt["indonesian"].str.split().str.len() >= 3)
]
df_mt = df_mt.drop_duplicates(subset=["indonesian", "dialect"]).reset_index(drop=True)

print(f"Filtered MT pairs: {len(df_mt):,}")
print(f"\nPer dialect:")
print(df_mt["dialect_name"].value_counts())
print(f"\nPer source:")
print(df_mt["source"].value_counts())


In [ ]:
from sklearn.model_selection import train_test_split

df_train_mt, df_val_mt = train_test_split(
    df_mt, test_size=0.05, random_state=RANDOM_SEED,
    stratify=df_mt["dialect_name"] if df_mt["dialect_name"].nunique() > 1 else None
)
df_train_mt = df_train_mt.reset_index(drop=True)
df_val_mt   = df_val_mt.reset_index(drop=True)

print(f"Train: {len(df_train_mt):,}  |  Val: {len(df_val_mt):,}")


## 6. Tokenization


In [ ]:
from transformers import MT5Tokenizer

tokenizer = MT5Tokenizer.from_pretrained(NORM_MODEL)
print(f"Loaded tokenizer: {NORM_MODEL}  vocab={len(tokenizer):,}")


def tokenize_pair(example):
    """Map function: tokenize dialect (input) and indonesian (target)."""
    inputs = tokenizer(
        example["dialect"],
        max_length=NORM_MAX_INPUT,
        truncation=True,
    )
    labels = tokenizer(
        text_target=example["indonesian"],
        max_length=NORM_MAX_TARGET,
        truncation=True,
    )
    # Label masking: padding token → -100 (HF DataCollator akan handle padding,
    # tapi kita tetap convert pad explicitly untuk safety)
    inputs["labels"] = [
        (tok if tok != tokenizer.pad_token_id else -100)
        for tok in labels["input_ids"]
    ]
    return inputs


In [ ]:
from datasets import Dataset

ds_train = Dataset.from_pandas(df_train_mt[["dialect", "indonesian"]])
ds_val   = Dataset.from_pandas(df_val_mt[["dialect",   "indonesian"]])

ds_train = ds_train.map(tokenize_pair, batched=False, remove_columns=ds_train.column_names)
ds_val   = ds_val.map(tokenize_pair,   batched=False, remove_columns=ds_val.column_names)

print(f"Tokenized train: {len(ds_train):,}")
print(f"Tokenized val  : {len(ds_val):,}")
print(f"Sample input_ids length: {len(ds_train[0]['input_ids'])}")
print(f"Sample labels   length : {len(ds_train[0]['labels'])}")


## 7. Load model + DataCollator


In [ ]:
from transformers import MT5ForConditionalGeneration, DataCollatorForSeq2Seq

model = MT5ForConditionalGeneration.from_pretrained(NORM_MODEL)
model.config.decoder_start_token_id = tokenizer.pad_token_id   # mT5 default

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model: {NORM_MODEL}  ({n_params:.1f}M params)")

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding="longest", return_tensors="pt")


## 8. Training arguments + Trainer


In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

args = Seq2SeqTrainingArguments(
    output_dir                  = NORM_OUTPUT_DIR,
    num_train_epochs            = NORM_EPOCHS,
    per_device_train_batch_size = NORM_BATCH,
    per_device_eval_batch_size  = NORM_BATCH,
    gradient_accumulation_steps = NORM_GRAD_ACCUM,
    learning_rate               = NORM_LR,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    max_grad_norm               = NORM_MAX_GRAD_NORM,
    bf16                        = USE_BF16,
    fp16                        = False,
    predict_with_generate       = True,
    generation_max_length       = NORM_MAX_TARGET,
    generation_num_beams        = 4,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    save_total_limit            = 2,
    label_smoothing_factor      = 0.1,
    gradient_checkpointing      = True,
    logging_steps               = 50,
    report_to                   = "none",
    seed                        = RANDOM_SEED,
)

trainer = Seq2SeqTrainer(
    model           = model,
    args            = args,
    train_dataset   = ds_train,
    eval_dataset    = ds_val,
    data_collator   = collator,
    tokenizer       = tokenizer,
)
print("Trainer ready.")


## 9. Train


In [ ]:
trainer.train()
trainer.save_model(NORM_OUTPUT_DIR)
tokenizer.save_pretrained(NORM_OUTPUT_DIR)
print(f"\nSaved normalizer → {NORM_OUTPUT_DIR}")


## 10. Evaluate — BLEU-4 + chrF++ on validation

Target: **BLEU-4 > 20** sebelum normalizer dipakai di pipeline.


In [ ]:
import sacrebleu

def evaluate_normalizer(model, tokenizer, val_df, n=200):
    model.eval()
    preds, refs = [], []
    sample = val_df.sample(min(n, len(val_df)), random_state=RANDOM_SEED)
    for _, row in sample.iterrows():
        inputs = tokenizer(row["dialect"], return_tensors="pt",
                           max_length=NORM_MAX_INPUT, truncation=True).to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=NORM_MAX_TARGET, num_beams=4,
                                 no_repeat_ngram_size=3, early_stopping=True)
        preds.append(tokenizer.decode(out[0], skip_special_tokens=True))
        refs.append(row["indonesian"])

    bleu = sacrebleu.corpus_bleu(preds, [refs])
    chrf = sacrebleu.corpus_chrf(preds, [refs])
    print(f"BLEU-4  : {bleu.score:.2f}  (target: > 20)")
    print(f"chrF++  : {chrf.score:.2f}")
    return {"bleu4": bleu.score, "chrf": chrf.score, "preds": preds, "refs": refs}


model.to(DEVICE)
eval_results = evaluate_normalizer(model, tokenizer, df_val_mt, n=200)

print("\nSample predictions:")
for i in range(min(5, len(eval_results["preds"]))):
    print(f"  DIAL : {df_val_mt.iloc[i]['dialect'][:80]}")
    print(f"  PRED : {eval_results['preds'][i][:80]}")
    print(f"  REF  : {eval_results['refs'][i][:80]}")
    print()


## 11. Bonus — Auto-expand dialect dictionary

Ekstrak pasangan kata dari NusaX untuk memperluas kamus preprocessing. Hasil disimpan di `data/dialect_dict.json` dan dapat dipakai oleh notebook lain sebagai fallback rule-based saat normalizer belum confident.


In [ ]:
def extract_dialect_pairs(df, dialect_col, bi_col="indonesian", top_n=300):
    pairs = {}
    if dialect_col not in df.columns:
        return pairs
    for _, row in df.iterrows():
        bi   = str(row.get(bi_col,    "") or "").lower().split()
        dial = str(row.get(dialect_col,"") or "").lower().split()
        if len(bi) != len(dial):
            continue
        for d, b in zip(dial, bi):
            if d != b and len(d) > 2 and len(b) > 2 and d.isalpha() and b.isalpha():
                pairs.setdefault(d, b)
        if len(pairs) >= top_n:
            break
    return pairs


dialect_dict = {}
if df_nusax is not None:
    for d in ["javanese", "sundanese", "minangkabau", "balinese", "banjarese",
              "madurese", "acehnese", "buginese", "ngaju", "toba_batak"]:
        if d in df_nusax.columns:
            pairs = extract_dialect_pairs(df_nusax, d, top_n=300)
            dialect_dict[d] = pairs
            print(f"  {d:14s} : {len(pairs):3d} pairs")

dict_path = DATA_DIR / "dialect_dict.json"
with open(dict_path, "w", encoding="utf-8") as f:
    json.dump(dialect_dict, f, ensure_ascii=False, indent=2)
print(f"\nSaved → {dict_path}")


## 12. Free VRAM


In [ ]:
def free_vram(*models):
    for m in models:
        del m
    gc.collect(); torch.cuda.empty_cache()
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM freed. Used: {used:.1f} / {total:.1f} GB")


free_vram(model, trainer)


## ✅ Selesai

Output:
- `models/normalizer/` — mT5-small checkpoint + tokenizer
- `data/dialect_dict.json` — kamus dialek hasil auto-expand

**Langkah selanjutnya:** jalankan `training_sum.ipynb` untuk training Stage 2 (3 summarizer).
